# Caderno de cálculos do ARTIGO 3

**Monitorando um banco de dados: painel de telemetria nativa do SQL Server e sinalização de problemas por limiares individuais**

Este caderno reproduz **todos os valores numéricos publicados no ARTIGO 3**, a partir dos arquivos de telemetria em `data/<instância>/` e dos logs de injeção de falhas. Cada bloco indica onde no artigo o resultado aparece.

Nada aqui se conecta a servidor: só são lidos os CSVs já extraídos pelo painel.

**Como ler este caderno.** Cada seção começa com uma nota curta dizendo em que parágrafo, quadro ou tabela do artigo o número aparece. Os blocos que sustentam as *conclusões* — seções 4.4 e 5 — vêm com uma explicação mais longa, porque ali o número sozinho não basta: é preciso entender por que ele significa o que o artigo afirma.

**Janela de reprodução.** A coleta continuou depois que o artigo foi escrito. Para reproduzir os valores publicados, a série é cortada na última coleta existente quando o texto foi fechado — 21 de setembro de 2026. Sem esse corte, os números mudam, porque a série cresceu.

In [1]:
import os
import re
import ast
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# Raiz do projeto do painel: a pasta que contém "data" e "app".
# Procura a partir da pasta onde este caderno foi aberto, subindo até três níveis,
# de modo que ele funcione tanto na raiz do projeto quanto em "escrita do artigo".
def achar_raiz(inicio):
    p = os.path.abspath(inicio)
    for _ in range(4):
        if os.path.isdir(os.path.join(p, "data")) and os.path.isdir(os.path.join(p, "app")):
            return p
        pai = os.path.dirname(p)
        if pai == p:
            break
        p = pai
    raise RuntimeError("Não encontrei a pasta do painel (a que contém 'data' e 'app'). "
                       "Abra este caderno de dentro do projeto ou defina RAIZ à mão.")

RAIZ = achar_raiz(os.getcwd())
DATA = os.path.join(RAIZ, "data")

# Ultima coleta de cada instancia quando o artigo foi fechado.
FIM_DO_ARTIGO = {
    "LAB-SQL": "2026-09-21 20:16",
    "SQLDEV":  "2026-09-21 20:18",
    "Prod":    "2026-09-21 20:21",
}
INSTANCIAS = ["LAB-SQL", "SQLDEV", "Prod"]
EXPERIMENTAIS = ["LAB-SQL", "SQLDEV"]

print("Projeto:", RAIZ)
print("Instâncias:", ", ".join(INSTANCIAS))

Projeto: /sessions/rcw-01fdzol5mm86mpcizk3a54j2/mnt/dba-monitor painel local
Instâncias: LAB-SQL, SQLDEV, Prod


## 1. Os limiares fixos vêm da própria aplicação

**Onde aparece no artigo:** Quadro 3 (seção 3.5) e todas as linhas "Limiar fixo" da Tabela 1.

Os limiares não são redigitados aqui. São lidos do dicionário `LIMITES` de `app/views/dashboard.py`, o mesmo que o painel usa para pintar os cartões. É o que a seção 3.6 quer dizer com *"a rotina importa os limiares e a função de sinalização dos próprios módulos da aplicação, medindo o sistema entregue"*: se alguém mudar um corte no painel, este caderno passa a medir o corte novo, e não uma cópia que envelheceu.

In [2]:
fonte = open(os.path.join(RAIZ, "app", "views", "dashboard.py"), encoding="utf-8").read()
LIMITES = ast.literal_eval(re.search(r"LIMITES\s*=\s*(\{.*?\n\})", fonte, re.S).group(1))

# Os quatro indicadores avaliados (secao 3.5): indicador do painel -> coluna da grade
COLS = {
    "ple_segundos":         "PageLifeExpectancy",
    "buffer_cache_hit_pct": "BufferCacheHitRatio",
    "cpu_sql_pct":          "SQLServerCPU",
    "sessoes_bloqueadas":   "SessoesBloqueadas",
}

pd.DataFrame(
    [(k, LIMITES[k]["aviso"], LIMITES[k]["critico"], LIMITES[k]["direcao"]) for k in LIMITES],
    columns=["indicador", "aviso", "crítico", "sentido"],
)

,indicador,aviso,crítico,sentido
0,ple_segundos,1000,300,abaixo
1,buffer_cache_hit_pct,97,95,abaixo
2,cpu_sql_pct,70,90,acima
3,cpu_total_pct,75,90,acima
4,latencia_leitura_ms,15,25,acima
5,latencia_escrita_ms,15,25,acima
6,sessoes_bloqueadas,1,5,acima
7,espaco_livre_pct,20,10,abaixo


## 2. Funções de base

**Onde aparece no artigo:** seção 3.4 (alinhamento em grade de um minuto) e seção 3.6 (matriz de confusão).

Quatro decisões de tratamento estão codificadas aqui, todas declaradas no texto:

- as métricas são **reamostradas em grade de um minuto pela média**, para que processador e atividade SQL possam ser comparados no mesmo instante;
- a **taxa de acerto do buffer cache é limitada à faixa de 0 a 100%**, por causa do artefato de leitura não simultânea medido na seção 4;
- um minuto é **problemático por construção** quando cai dentro de uma injeção com status `CONCLUIDA` — injeções que terminaram em erro não rotulam nada;
- o alarme de um arranjo é a **união** dos indicadores: basta um cruzar seu corte.

In [3]:
def grade(instancia, ate=None):
    '''Telemetria de processador e atividade em grade de um minuto.'''
    cpu = pd.read_csv(
        os.path.join(DATA, instancia, "cpu_metrics.csv"),
        usecols=["DataColeta", "SQLServerCPU", "PageLifeExpectancy", "BufferCacheHitRatio"])
    ativ = pd.read_csv(
        os.path.join(DATA, instancia, "activity_metrics.csv"),
        usecols=["DataColeta", "SessoesBloqueadas", "ConexoesAtivas",
                 "BatchRequestsPeriodo", "DeadlocksPeriodo"])
    for d in (cpu, ativ):
        d["DataColeta"] = pd.to_datetime(d["DataColeta"], errors="coerce")
        d.dropna(subset=["DataColeta"], inplace=True)

    g = (cpu.set_index("DataColeta").resample("1min").mean()
            .join(ativ.set_index("DataColeta").resample("1min").mean(), how="outer")
            .sort_index())
    g["BufferCacheHitRatio"] = g["BufferCacheHitRatio"].clip(0, 100)
    g = g.dropna(how="all")
    if ate is None:
        ate = FIM_DO_ARTIGO.get(instancia)
    return g[g.index <= ate] if ate else g


def injecoes(instancia):
    '''Log de injeções concluídas — a verdade de referência.'''
    d = pd.read_csv(os.path.join(DATA, instancia, "injections.csv"),
                    sep=";", encoding="utf-8-sig")
    d.columns = [c.replace("﻿", "").strip() for c in d.columns]
    d = d[d["Status"].astype(str).str.upper() == "CONCLUIDA"].copy()
    d["DataInicio"] = pd.to_datetime(d["DataInicio"])
    d["DataFim"] = pd.to_datetime(d["DataFim"])
    return d.sort_values("DataInicio").reset_index(drop=True)


def rotular(g, inj):
    '''Marca cada minuto da grade como problemático ou não, e com que modo de falha.'''
    verdade = pd.Series(False, index=g.index)
    modo = pd.Series(None, index=g.index, dtype=object)
    for _, r in inj.iterrows():
        dentro = (g.index >= r["DataInicio"].floor("min")) & (g.index <= r["DataFim"].floor("min"))
        verdade[dentro] = True
        modo[dentro] = r["ModoFalha"]
    return verdade, modo


def alarme(g, nivel, calibrados=None):
    '''União dos indicadores que cruzaram seu corte no nível pedido.'''
    m = pd.Series(False, index=g.index)
    for chave, col in COLS.items():
        cfg = LIMITES[chave]
        if calibrados is not None:
            if calibrados.get(chave) is None:
                continue                      # indicador sem dispersão: descartado
            corte = calibrados[chave][nivel]
        else:
            corte = cfg[nivel]
        x = g[col]
        m |= (x < corte) if cfg["direcao"] == "abaixo" else (x > corte)
    return m.fillna(False)


def medidas(verdade, alarmes):
    '''Matriz de confusão minuto a minuto e as quatro medidas da seção 3.6.'''
    vp = int((verdade & alarmes).sum())
    fp = int((~verdade & alarmes).sum())
    fn = int((verdade & ~alarmes).sum())
    vn = int((~verdade & ~alarmes).sum())
    precisao = vp / (vp + fp) if vp + fp else 0.0
    revocacao = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precisao * revocacao / (precisao + revocacao) if precisao + revocacao else 0.0
    tfp = fp / (fp + vn) if fp + vn else 0.0
    return dict(VP=vp, FP=fp, FN=fn, VN=vn, Precisao=precisao,
                Revocacao=revocacao, F1=f1, TFP=tfp)


def linha_de_base(g, inj):
    '''Trecho anterior à primeira injeção — onde os percentis são calculados.'''
    return g[g.index < inj["DataInicio"].min().floor("min")]


def calibrar(base):
    '''Percentis 5/1 para desvio para baixo e 95/99 para cima (seção 3.5).'''
    cal = {}
    for chave, col in COLS.items():
        v = base[col].dropna()
        if v.nunique() <= 1:
            cal[chave] = None                 # variância nula: não há o que calibrar
            continue
        if LIMITES[chave]["direcao"] == "abaixo":
            cal[chave] = {"aviso": float(np.percentile(v, 5)),
                          "critico": float(np.percentile(v, 1))}
        else:
            cal[chave] = {"aviso": float(np.percentile(v, 95)),
                          "critico": float(np.percentile(v, 99))}
    return cal

print("funções definidas")

funções definidas


---
## 3. Seção 3.2 — Período, cobertura e linha de base

**Onde aparece no artigo:** parágrafo *"A coleta iniciou-se em 2 de setembro..."*, com os valores 10.184 e 10.903 minutos de linha de base, 26.516 minutos em produção, cobertura de 100,0% / 100,0% / 99,9% e a interrupção única de 17 minutos.

A cobertura compara os minutos efetivamente coletados com os minutos que deveriam existir entre a primeira e a última amostra. A diferença é o número de minutos sem coleta.

In [4]:
linhas = []
for inst in INSTANCIAS:
    g = grade(inst)
    esperados = int((g.index.max() - g.index.min()).total_seconds() // 60) + 1
    if inst in EXPERIMENTAIS:
        inj = injecoes(inst)
        primeira = inj["DataInicio"].min()
        base = int((g.index <= primeira.floor("min")).sum())
    else:
        primeira, base = None, len(g)      # produção não recebeu injeção
    linhas.append(dict(
        Instancia=inst,
        Inicio=g.index.min(), Fim=g.index.max(),
        Minutos=len(g), Esperados=esperados,
        Cobertura="%.1f%%" % (len(g) / esperados * 100),
        Lacunas=esperados - len(g),
        PrimeiraInjecao=primeira,
        LinhaDeBase=base))

periodo = pd.DataFrame(linhas)
periodo

,Instancia,Inicio,Fim,Minutos,Esperados,Cobertura,Lacunas,PrimeiraInjecao,LinhaDeBase
0,LAB-SQL,2026-09-02 22:17:00,2026-09-21 20:16:00,27229,27240,100.0%,11,2026-09-10 00:00:01,10184
1,SQLDEV,2026-09-02 22:33:00,2026-09-21 20:18:00,27218,27226,100.0%,8,2026-09-10 12:15:01,10903
2,Prod,2026-09-03 10:09:00,2026-09-21 20:21:00,26516,26533,99.9%,17,NaT,26516


> **Nota de convenção.** A linha de base acima inclui o minuto em que a primeira injeção começou — é o número publicado na seção 3.2 (10.184 e 10.903). O cálculo dos percentis, mais adiante, usa estritamente os minutos **anteriores** a esse instante, um minuto a menos, para que nenhuma amostra já degradada entre no limiar derivado.

## 4. Quadro 1 — porte de cada instância

**Onde aparece no artigo:** Quadro 1, coluna "Bancos / arq.".

Contagem feita sobre a última coleta de entrada e saída dentro da janela: cada linha dessa coleta é um arquivo de banco de dados.

In [5]:
portes = []
for inst in INSTANCIAS:
    io = pd.read_csv(os.path.join(DATA, inst, "io_stats.csv"),
                     usecols=["DataColeta", "DatabaseName", "FileName"])
    io["DataColeta"] = pd.to_datetime(io["DataColeta"], errors="coerce")
    io = io[io["DataColeta"] <= FIM_DO_ARTIGO[inst]]
    ultima = io[io["DataColeta"] == io["DataColeta"].max()]
    portes.append(dict(Instancia=inst, Coleta=io["DataColeta"].max(),
                       Bancos=ultima["DatabaseName"].nunique(), Arquivos=len(ultima)))

portes = pd.DataFrame(portes)
portes["PublicadoNoArtigo"] = ["12 / 32", "59 / 134", "69 / 154"]
portes

,Instancia,Coleta,Bancos,Arquivos,PublicadoNoArtigo
0,LAB-SQL,2026-09-21 20:16:00,12,32,12 / 32
1,SQLDEV,2026-09-21 20:17:00,61,138,59 / 134
2,Prod,2026-09-21 20:20:01,70,156,69 / 154


> **Divergência.** LAB-SQL confere. Em SQLDEV e em produção há hoje mais bancos do que o Quadro 1 registra, porque o quadro foi montado com a extração de meados de setembro e bases novas foram criadas depois. Os números do quadro descrevem o ambiente na data em que o artigo o descreveu; os desta célula, o ambiente na data da última extração.

## 5. Seção 3.4 — o artefato da taxa de acerto do buffer cache

**Onde aparece no artigo:** parágrafo *"Dois tratamentos são necessários: a taxa de acerto do buffer cache excedeu 100%..."*.

O contador é uma razão entre dois acumuladores lidos em instantes ligeiramente diferentes; quando o denominador é lido depois de o numerador já ter avançado, a razão estoura. Por isso a série é limitada à faixa de 0 a 100% na função `grade`.

In [6]:
registros = 0
acima = 0
maximo = 0.0
por_instancia = []
for inst in INSTANCIAS:
    d = pd.read_csv(os.path.join(DATA, inst, "cpu_metrics.csv"),
                    usecols=["DataColeta", "BufferCacheHitRatio"])
    d["DataColeta"] = pd.to_datetime(d["DataColeta"], errors="coerce")
    d = d[d["DataColeta"] <= FIM_DO_ARTIGO[inst]]
    n = len(d)
    a = int((d["BufferCacheHitRatio"] > 100).sum())
    por_instancia.append(dict(Instancia=inst, Registros=n, AcimaDe100=a,
                              Maximo=round(d["BufferCacheHitRatio"].max(), 1)))
    registros += n
    acima += a
    maximo = max(maximo, d["BufferCacheHitRatio"].max())

print("Total: %d registros, %d acima de 100%%, máximo de %.1f%%" % (registros, acima, maximo))
pd.DataFrame(por_instancia)

Total: 80964 registros, 6 acima de 100%, máximo de 904.9%


,Instancia,Registros,AcimaDe100,Maximo
0,LAB-SQL,27230,0,100.0
1,SQLDEV,27218,5,822.0
2,Prod,26516,1,904.9


> **Divergência.** O artigo diz "quatro dos 22.670 registros, chegando a 904,9%". O máximo confere, mas a contagem foi feita sobre uma extração anterior e menor. Sobre a série completa das três instâncias na janela do artigo, são seis registros em 80.963. A afirmação continua verdadeira no essencial — o artefato é raro e o tratamento é necessário —, mas os dois números precisam ser corrigidos no texto.

## 6. Seção 3.6 — as injeções que formam a verdade de referência

**Onde aparece no artigo:** resumo (98 injeções no laboratório e 88 em desenvolvimento), seção 4.4 (16 injeções encerradas em erro, descartadas) e seção 4.4 (comparação entre as duas campanhas).

A segunda campanha, a partir de 18 de setembro, é a que passou a dimensionar a carga por paralelismo — um quarto, metade e totalidade dos processadores lógicos. A separação por data é o que permite afirmar, na discussão, que parte do resultado da primeira campanha vinha do instrumento de medida, e não do sinalizador.

In [7]:
resumo_inj = []
for inst in EXPERIMENTAIS:
    bruto = pd.read_csv(os.path.join(DATA, inst, "injections.csv"), sep=";", encoding="utf-8-sig")
    bruto.columns = [c.replace("﻿", "").strip() for c in bruto.columns]
    status = bruto["Status"].astype(str).str.upper().value_counts().to_dict()
    inj = injecoes(inst)
    resumo_inj.append(dict(
        Instancia=inst,
        Concluidas=status.get("CONCLUIDA", 0),
        Erro=status.get("ERRO", 0),
        PrimeiraCampanha=int((inj["DataInicio"] < "2026-09-18").sum()),
        SegundaCampanha=int((inj["DataInicio"] >= "2026-09-18").sum()),
        Modos=inj["ModoFalha"].nunique(),
        DuracaoMediaSeg=round((inj["DataFim"] - inj["DataInicio"]).dt.total_seconds().mean(), 1)))

pd.DataFrame(resumo_inj)

,Instancia,Concluidas,Erro,PrimeiraCampanha,SegundaCampanha,Modos,DuracaoMediaSeg
0,LAB-SQL,98,0,60,38,4,181.8
1,SQLDEV,88,16,50,38,4,180.1


---
## 7. Quadro 4 — a linha de base das três instâncias

**Onde aparece no artigo:** Quadro 4 (seção 4.2), e os valores citados no texto: expectativa de vida de página de 1.298.246 segundos no laboratório, 864 segundos em produção, e a queda para 16 segundos sob carga injetada.

Todas as medianas são da linha de base — antes de qualquer injeção. Em produção, que não recebeu injeção, a linha de base é o período inteiro.

In [8]:
METRICAS_QUADRO4 = [
    ("SQLServerCPU",         "CPU do SQL Server (%)",               "mediana"),
    ("SQLServerCPU",         "CPU máxima observada (%)",            "maximo"),
    ("BatchRequestsPeriodo", "Requisições de lote por minuto",      "mediana"),
    ("ConexoesAtivas",       "Conexões ativas",                     "mediana"),
    ("ConexoesAtivas",       "Pico de conexões",                    "maximo"),
    ("PageLifeExpectancy",   "Expectativa de vida de página (s)",   "mediana"),
    ("SessoesBloqueadas",    "Máximo de sessões bloqueadas",        "maximo"),
    ("DeadlocksPeriodo",     "Máximo de deadlocks por minuto",      "maximo"),
]

quadro4 = {}
for inst in INSTANCIAS:
    g = grade(inst)
    base = g[g.index <= injecoes(inst)["DataInicio"].min().floor("min")] if inst in EXPERIMENTAIS else g
    coluna = {}
    for col, rotulo, como in METRICAS_QUADRO4:
        v = base[col].dropna()
        coluna[rotulo] = round(v.median() if como == "mediana" else v.max())
    # tipos de espera distintos
    w = pd.read_csv(os.path.join(DATA, inst, "wait_stats.csv"), usecols=["DataColeta", "WaitType"])
    w["DataColeta"] = pd.to_datetime(w["DataColeta"], errors="coerce")
    coluna["Tipos de espera distintos"] = w[w["DataColeta"] <= FIM_DO_ARTIGO[inst]]["WaitType"].nunique()
    quadro4[inst] = coluna

pd.DataFrame(quadro4).astype("int64")

,LAB-SQL,SQLDEV,Prod
CPU do SQL Server (%),0,5,28
CPU máxima observada (%),28,53,97
Requisições de lote por minuto,38,62504,342228
Conexões ativas,6,193,918
Pico de conexões,23,293,2204
Expectativa de vida de página (s),1298246,1633,864
Máximo de sessões bloqueadas,1,3,14
Máximo de deadlocks por minuto,0,0,19
Tipos de espera distintos,65,127,133


In [9]:
# A queda da expectativa de vida de página no laboratório sob carga injetada
g = grade("LAB-SQL")
inj = injecoes("LAB-SQL")
verdade, modo = rotular(g, inj)
base = g[g.index <= inj["DataInicio"].min().floor("min")]
print("LAB-SQL — expectativa de vida de página")
print("   mediana na linha de base : %.0f s  (cerca de %.0f dias)"
      % (base["PageLifeExpectancy"].median(), base["PageLifeExpectancy"].median() / 86400))
print("   mínimo dentro de injeção : %.0f s" % g.loc[verdade, "PageLifeExpectancy"].min())

LAB-SQL — expectativa de vida de página
   mediana na linha de base : 1298246 s  (cerca de 15 dias)
   mínimo dentro de injeção : 16 s


## 8. Seção 4.2 — o ciclo de utilização da produção

**Onde aparece no artigo:** parágrafo *"O ciclo de utilização da produção revela outra particularidade..."*: processador entre 22% e 36% de mediana por hora, razão de 1,64 entre pico e vale, 1.647 conexões na faixa das 12h às 17h contra 754 na madrugada, e 1.054 em dias úteis contra 409 nos fins de semana.

Este é o bloco que sustenta a afirmação de que **a sazonalidade da produção se manifesta na concorrência, e não no consumo de processador** — e, por consequência, que um limiar aplicado isoladamente ao processador seria cego a esse ciclo.

In [10]:
g = grade("Prod").copy()
g["hora"] = g.index.hour
g["dia_semana"] = g.index.dayofweek
faixa = pd.cut(g["hora"], [-1, 5, 11, 17, 23], labels=["00-05h", "06-11h", "12-17h", "18-23h"])

por_faixa = pd.DataFrame({
    "Conexões ativas (mediana)": g.groupby(faixa, observed=False)["ConexoesAtivas"].median().round(0),
    "CPU do SQL Server (mediana %)": g.groupby(faixa, observed=False)["SQLServerCPU"].median().round(1),
})
print(por_faixa.to_string(), "\n")

cpu_hora = g.groupby("hora")["SQLServerCPU"].median()
print("CPU mediana por hora: vale %.0f%% | pico %.0f%% | razão pico/vale %.2f"
      % (cpu_hora.min(), cpu_hora.max(), cpu_hora.max() / cpu_hora.min()))

conexoes_hora = g.groupby("hora")["ConexoesAtivas"].median()
print("Conexões medianas por hora: vale %.0f | pico %.0f | razão pico/vale %.2f"
      % (conexoes_hora.min(), conexoes_hora.max(), conexoes_hora.max() / conexoes_hora.min()))

print("Conexões: dias úteis %.0f | fins de semana %.0f"
      % (g[g["dia_semana"] < 5]["ConexoesAtivas"].median(),
         g[g["dia_semana"] >= 5]["ConexoesAtivas"].median()))

        Conexões ativas (mediana)  CPU do SQL Server (mediana %)
hora                                                            
00-05h                      754.0                           25.0
06-11h                      950.0                           31.0
12-17h                     1647.0                           31.0
18-23h                      910.0                           25.0 

CPU mediana por hora: vale 22% | pico 36% | razão pico/vale 1.64
Conexões medianas por hora: vale 681 | pico 1757 | razão pico/vale 2.58
Conexões: dias úteis 1054 | fins de semana 409


## 9. Seção 4.2 — as esperas que nenhum limiar captura

**Onde aparece no artigo:** parágrafo final da seção 4.2 — *"em produção, as duas maiores em tempo acumulado após a espera ociosa de escalonador são CXSYNC_PORT e CXCONSUMER"*.

`SOS_WORK_DISPATCHER` é espera ociosa de escalonador: aparece em primeiro lugar em qualquer instância e não indica problema. Descartada ela, as duas seguintes dizem respeito à sincronização de planos paralelos — informação que nenhum dos oito indicadores do Quadro 3 carrega, e que é o exemplo do artigo para o que a observabilidade oferece além da sinalização.

In [11]:
w = pd.read_csv(os.path.join(DATA, "Prod", "wait_stats.csv"),
                usecols=["DataColeta", "WaitType", "DeltaWaitTimeMs"])
w["DataColeta"] = pd.to_datetime(w["DataColeta"], errors="coerce")
w = w[w["DataColeta"] <= FIM_DO_ARTIGO["Prod"]]

total = (w.groupby("WaitType")["DeltaWaitTimeMs"].sum()
          .sort_values(ascending=False).head(8)
          .rename("Tempo de espera acumulado (ms)").to_frame())
total["Participação"] = (total.iloc[:, 0] / w["DeltaWaitTimeMs"].sum() * 100).round(2).astype(str) + "%"
total

,Tempo de espera acumulado (ms),Participação
WaitType,,
SOS_WORK_DISPATCHER,385913975178,94.05%
CXSYNC_PORT,3115265744,0.76%
CXCONSUMER,1806675330,0.44%
FT_IFTSHC_MUTEX,1608407673,0.39%
HADR_FILESTREAM_IOMGR_IOCOMPLETION,1591658949,0.39%
DIRTY_PAGE_POLL,1591607523,0.39%
SP_SERVER_DIAGNOSTICS_SLEEP,1591535597,0.39%
PWAIT_EXTENSIBILITY_CLEANUP_TASK,1591483928,0.39%


---
# Seção 4.3 — a medição central do artigo

As três células seguintes produzem as Tabelas 1, 2 e 3. É o núcleo do trabalho: cada arranjo de decisão é aplicado à mesma grade de minutos e confrontado com a mesma verdade de referência, de modo que as diferenças entre as linhas vêm só da regra, e não dos dados.

In [12]:
# Avaliação completa das duas instâncias experimentais.
avaliacao = {}
for inst in EXPERIMENTAIS:
    g = grade(inst)
    inj = injecoes(inst)
    verdade, modo = rotular(g, inj)
    base = linha_de_base(g, inj)          # estritamente antes da primeira injeção
    cal = calibrar(base)

    arranjos = {
        "Limiar fixo (aviso)":        alarme(g, "aviso"),
        "Limiar fixo (crítico)":      alarme(g, "critico"),
        "Limiar calibrado (aviso)":   alarme(g, "aviso", cal),
        "Limiar calibrado (crítico)": alarme(g, "critico", cal),
    }
    avaliacao[inst] = dict(grade=g, inj=inj, verdade=verdade, modo=modo,
                           base=base, calibrados=cal, arranjos=arranjos)

    print("%-8s %6d minutos | base %5d | %3d minutos com problema (%.2f%%) | %d injeções"
          % (inst, len(g), len(base), int(verdade.sum()),
             verdade.sum() / len(g) * 100, len(inj)))

LAB-SQL   27229 minutos | base 10183 | 389 minutos com problema (1.43%) | 98 injeções


SQLDEV    27218 minutos | base 10902 | 344 minutos com problema (1.26%) | 88 injeções


### Tabela 1 — capacidade de sinalização

**Onde aparece no artigo:** Tabela 1, e os números repetidos no resumo, na introdução, na seção 4.3, na discussão e nas considerações finais.

A prevalência de problemas é de 1,43% no laboratório e 1,26% em desenvolvimento. É por isso que a seção 4.4 argumenta que a acurácia global não serve: uma regra que nunca alarmasse acertaria quase 99% dos minutos.

In [13]:
tabela1 = []
for inst in EXPERIMENTAIS:
    a = avaliacao[inst]
    for nome, alarmes in a["arranjos"].items():
        m = medidas(a["verdade"], alarmes)
        tabela1.append(dict(
            Instancia=inst, Regra=nome,
            AlarmesFalsos=m["FP"],
            Precisao="%.2f%%" % (m["Precisao"] * 100),
            Revocacao="%.2f%%" % (m["Revocacao"] * 100),
            F1="%.2f%%" % (m["F1"] * 100),
            TFP="%.2f%%" % (m["TFP"] * 100)))

pd.DataFrame(tabela1)

,Instancia,Regra,AlarmesFalsos,Precisao,Revocacao,F1,TFP
0,LAB-SQL,Limiar fixo (aviso),67,43.70%,13.37%,20.47%,0.25%
1,LAB-SQL,Limiar fixo (crítico),19,48.65%,4.63%,8.45%,0.07%
2,LAB-SQL,Limiar calibrado (aviso),15343,2.41%,97.43%,4.70%,57.16%
3,LAB-SQL,Limiar calibrado (crítico),14935,2.47%,97.43%,4.83%,55.64%
4,SQLDEV,Limiar fixo (aviso),9080,1.61%,43.31%,3.11%,33.79%
5,SQLDEV,Limiar fixo (crítico),3284,2.00%,19.48%,3.63%,12.22%
6,SQLDEV,Limiar calibrado (aviso),3543,4.29%,46.22%,7.86%,13.18%
7,SQLDEV,Limiar calibrado (crítico),784,11.51%,29.65%,16.59%,2.92%


### Tabela 2 — revocação por modo de falha

**Onde aparece no artigo:** Tabela 2, e o parágrafo que descreve a saturação de entrada e saída como o único modo em que a calibração piora o resultado — de 37,50% para 26,04% em SQLDEV.

Essa piora é coerente com a limitação declarada na seção 4.4: as latências de disco ficaram fora dos quatro indicadores avaliados, então o modo de falha que ataca o disco precisa ser sinalizado por via indireta.

In [14]:
ROTULO_MODO = {"BLOQUEIO": "Bloqueio", "CPU": "Processador",
               "IO": "Entrada e saída", "MEMORIA": "Memória"}

tabela2 = []
for inst in EXPERIMENTAIS:
    a = avaliacao[inst]
    for md_, rotulo in ROTULO_MODO.items():
        sel = a["verdade"] & (a["modo"] == md_)
        n = int(sel.sum())
        if not n:
            continue
        tabela2.append(dict(
            Instancia=inst, Modo=rotulo, Minutos=n,
            LimiarFixo="%.2f%%" % ((sel & a["arranjos"]["Limiar fixo (aviso)"]).sum() / n * 100),
            LimiarCalibrado="%.2f%%" % ((sel & a["arranjos"]["Limiar calibrado (aviso)"]).sum() / n * 100)))

pd.DataFrame(tabela2)

,Instancia,Modo,Minutos,LimiarFixo,LimiarCalibrado
0,LAB-SQL,Bloqueio,99,39.39%,96.97%
1,LAB-SQL,Processador,98,9.18%,95.92%
2,LAB-SQL,Entrada e saída,96,1.04%,96.88%
3,LAB-SQL,Memória,96,3.12%,100.00%
4,SQLDEV,Bloqueio,73,53.42%,67.12%
5,SQLDEV,Processador,107,22.43%,27.10%
6,SQLDEV,Entrada e saída,96,37.50%,26.04%
7,SQLDEV,Memória,68,73.53%,82.35%


### Tabela 3 — os limiares derivados de cada instância

**Onde aparece no artigo:** Tabela 3, e os parágrafos que comentam a distância entre o corte derivado e o recomendado pela literatura: a expectativa de vida de página cai de 1.000 para 81 segundos em SQLDEV, e o processador, de 70% para 16%.

In [15]:
ROTULO_IND = {
    "ple_segundos": "Expectativa de vida de página (s)",
    "buffer_cache_hit_pct": "Taxa de acerto do buffer cache (%)",
    "cpu_sql_pct": "CPU do SQL Server (%)",
    "sessoes_bloqueadas": "Sessões bloqueadas",
}

tabela3 = []
for chave, rotulo in ROTULO_IND.items():
    linha = {"Indicador": rotulo,
             "Sentido": LIMITES[chave]["direcao"],
             "Fixo": "%s / %s" % (LIMITES[chave]["aviso"], LIMITES[chave]["critico"])}
    for inst in EXPERIMENTAIS:
        c = avaliacao[inst]["calibrados"][chave]
        linha[inst] = "descartado" if c is None else "%.2f / %.2f" % (c["aviso"], c["critico"])
    tabela3.append(linha)

pd.DataFrame(tabela3)

,Indicador,Sentido,Fixo,LAB-SQL,SQLDEV
0,Expectativa de vida de página (s),abaixo,1000 / 300,1023315.00 / 998878.38,81.00 / 20.00
1,Taxa de acerto do buffer cache (%),abaixo,97 / 95,100.00 / 100.00,99.76 / 97.89
2,CPU do SQL Server (%),acima,70 / 90,0.00 / 0.00,16.00 / 21.00
3,Sessões bloqueadas,acima,1 / 5,0.00 / 0.00,0.00 / 0.00


### Tempo entre o início da injeção e o primeiro alarme

**Onde aparece no artigo:** final da seção 4.3 — *"o limiar calibrado no nível de aviso sinaliza 60 das 88 injeções, contra 48 do limiar fixo, ambos com mediana de um minuto"* — e, para o laboratório, *"passa a sinalizar as 98 injeções, com mediana de zero minuto"*.

Mediana de zero minuto significa que o alarme já estava ativo no primeiro minuto da injeção. No laboratório calibrado isso é menos virtude do que sintoma: o alarme estava ativo quase sempre, inclusive fora das injeções, como a taxa de falsos positivos de 57,16% mostra.

In [16]:
latencias = []
for inst in EXPERIMENTAIS:
    a = avaliacao[inst]
    for nome, alarmes in a["arranjos"].items():
        atrasos = []
        for _, r in a["inj"].iterrows():
            ini, fim = r["DataInicio"].floor("min"), r["DataFim"].floor("min")
            janela = alarmes[(alarmes.index >= ini) & (alarmes.index <= fim)]
            disparou = janela[janela]
            if len(disparou):
                atrasos.append((disparou.index[0] - ini).total_seconds() / 60)
        latencias.append(dict(
            Instancia=inst, Regra=nome,
            Injecoes=len(a["inj"]), Sinalizadas=len(atrasos),
            AtrasoMedio=round(float(np.mean(atrasos)), 2) if atrasos else None,
            AtrasoMediano=round(float(np.median(atrasos)), 1) if atrasos else None))

pd.DataFrame(latencias)

,Instancia,Regra,Injecoes,Sinalizadas,AtrasoMedio,AtrasoMediano
0,LAB-SQL,Limiar fixo (aviso),98,20,1.25,1.0
1,LAB-SQL,Limiar fixo (crítico),98,7,1.29,1.0
2,LAB-SQL,Limiar calibrado (aviso),98,98,0.10,0.0
3,LAB-SQL,Limiar calibrado (crítico),98,98,0.10,0.0
4,SQLDEV,Limiar fixo (aviso),88,48,0.67,1.0
5,SQLDEV,Limiar fixo (crítico),88,26,1.08,1.0
6,SQLDEV,Limiar calibrado (aviso),88,60,1.12,1.0
7,SQLDEV,Limiar calibrado (crítico),88,40,1.23,1.0


---
# Seção 4.4 e Seção 5 — os cálculos que sustentam as conclusões

As células até aqui produziram números. As três a seguir produzem **argumentos**: cada uma testa uma das afirmações conclusivas do artigo. São as que merecem leitura mais atenta, porque nelas o valor calculado só significa alguma coisa junto com o raciocínio que o cerca.

## 10. Primeira conclusão — limiares absolutos não são transferíveis

**Onde aparece no artigo:** primeiro parágrafo da seção 4.4 e segundo parágrafo das considerações finais.

A afirmação é forte: *o mesmo conjunto de regras alarma em 0,25% dos minutos de uma instância e em 33,79% dos minutos de outra, sem que nenhuma regra tenha mudado*. Para que isso seja um resultado, e não uma coincidência, três condições precisam valer ao mesmo tempo, e a célula abaixo verifica as três.

A primeira é que **as regras sejam literalmente as mesmas**: os cortes vêm do mesmo dicionário `LIMITES`, carregado uma única vez, e a função de alarme é a mesma chamada.

A segunda é que **a verdade de referência seja comparável**: as duas instâncias receberam o mesmo delineamento fatorial, quatro modos por três intensidades, e a prevalência de minutos com problema é próxima — 1,43% e 1,26%. Se uma tivesse dez vezes mais minutos degradados que a outra, a diferença nas taxas de alarme poderia vir daí.

A terceira é que **a diferença esteja na instância, e não no acaso da janela**: as duas séries têm tamanho quase idêntico, cobrem o mesmo período de calendário e incluem os mesmos fins de semana.

Satisfeitas as três, resta uma explicação: o corte fixo está posicionado em lugares completamente diferentes da distribuição de cada instância. A última coluna da célula mostra isso diretamente — qual fração da linha de base de cada instância já cruza, sozinha, cada limiar fixo, sem nenhuma falha acontecendo.

In [17]:
print("Mesmas regras, instâncias diferentes\n")
comparacao = []
for inst in EXPERIMENTAIS:
    a = avaliacao[inst]
    m = medidas(a["verdade"], a["arranjos"]["Limiar fixo (aviso)"])
    comparacao.append(dict(
        Instancia=inst,
        Minutos=len(a["grade"]),
        MinutosComProblema=int(a["verdade"].sum()),
        Prevalencia="%.2f%%" % (a["verdade"].sum() / len(a["grade"]) * 100),
        Revocacao="%.2f%%" % (m["Revocacao"] * 100),
        TaxaFalsosPositivos="%.2f%%" % (m["TFP"] * 100),
        AlarmesFalsos=m["FP"]))
print(pd.DataFrame(comparacao).to_string(index=False), "\n")

print("Onde o limiar fixo cai na linha de base de cada instância")
print("(fração dos minutos SEM falha alguma que já cruzam o corte de aviso)\n")
posicao = []
for inst in EXPERIMENTAIS:
    base = avaliacao[inst]["base"]
    for chave, col in COLS.items():
        cfg = LIMITES[chave]
        x = base[col].dropna()
        cruza = (x < cfg["aviso"]).mean() if cfg["direcao"] == "abaixo" else (x > cfg["aviso"]).mean()
        posicao.append(dict(Instancia=inst, Indicador=ROTULO_IND[chave],
                            LimiarFixo=cfg["aviso"],
                            MedianaDaBase=round(float(x.median()), 2),
                            CruzaNaBase="%.2f%%" % (cruza * 100)))
pd.DataFrame(posicao)

Mesmas regras, instâncias diferentes

Instancia  Minutos  MinutosComProblema Prevalencia Revocacao TaxaFalsosPositivos  AlarmesFalsos
  LAB-SQL    27229                 389       1.43%    13.37%               0.25%             67
   SQLDEV    27218                 344       1.26%    43.31%              33.79%           9080 

Onde o limiar fixo cai na linha de base de cada instância
(fração dos minutos SEM falha alguma que já cruzam o corte de aviso)



,Instancia,Indicador,LimiarFixo,MedianaDaBase,CruzaNaBase
0,LAB-SQL,Expectativa de vida de página (s),1000,1298216.0,0.00%
1,LAB-SQL,Taxa de acerto do buffer cache (%),97,100.0,0.03%
2,LAB-SQL,CPU do SQL Server (%),70,0.0,0.00%
3,LAB-SQL,Sessões bloqueadas,1,0.0,0.00%
4,SQLDEV,Expectativa de vida de página (s),1000,1633.0,34.55%
5,SQLDEV,Taxa de acerto do buffer cache (%),97,100.0,0.67%
6,SQLDEV,CPU do SQL Server (%),70,5.0,0.00%
7,SQLDEV,Sessões bloqueadas,1,0.0,0.06%


> **Leitura.** Em SQLDEV, um único indicador — a expectativa de vida de página — já cruza o limiar fixo de aviso em cerca de um terço dos minutos da linha de base, quando ainda não havia injeção nenhuma. É daí que vêm os 33,79% de taxa de falsos positivos: não de uma instância que falha muito, mas de um corte posicionado dentro do comportamento normal dela. No laboratório, o mesmo corte não é cruzado em minuto algum da base, e o alarme quase nunca dispara. A mesma régua, dois resultados opostos.

> **Correção a fazer no texto.** A seção 4.4 atribui esse alarme contínuo à *taxa de acerto do buffer cache*. A tabela acima mostra que o indicador responsável é a *expectativa de vida de página*: ela cruza o corte em cerca de um terço dos minutos da base de SQLDEV, enquanto a taxa de acerto do buffer cache cruza em menos de 1%. A conclusão do parágrafo não muda — o limiar fixo estava mal posicionado nos dois sentidos ao mesmo tempo —, mas o indicador citado precisa ser trocado.

## 11. Segunda conclusão — a calibração melhora, onde há carga

**Onde aparece no artigo:** segundo parágrafo da seção 4.4 e segundo parágrafo das considerações finais.

O artigo destaca que, em SQLDEV, a calibração melhorou **simultaneamente** a revocação e a taxa de falsos positivos, e observa que isso é incomum, pois as duas medidas costumam ser antagônicas: baixar um corte para capturar mais problemas normalmente traz junto mais alarme falso.

A explicação oferecida é que o limiar fixo estava mal posicionado **nos dois sentidos ao mesmo tempo** — baixo demais para um indicador e alto demais para outro. Quando isso acontece, mover cada corte para o lugar certo pode melhorar os dois eixos de uma vez, porque não se está trocando um pelo outro: está-se corrigindo dois erros independentes.

A célula abaixo quantifica esse ganho duplo e, em seguida, mostra de onde ele vem, decompondo o alarme por indicador: quantos minutos cada um faz disparar sozinho, sob cada regra.

In [18]:
inst = "SQLDEV"
a = avaliacao[inst]
print("Ganho da calibração em %s\n" % inst)
for nivel in ["aviso", "critico"]:
    fixo = medidas(a["verdade"], a["arranjos"]["Limiar fixo (%s)" % ("aviso" if nivel == "aviso" else "crítico")])
    cal = medidas(a["verdade"], a["arranjos"]["Limiar calibrado (%s)" % ("aviso" if nivel == "aviso" else "crítico")])
    print("  nível %s" % nivel)
    for rotulo, chave, pct in [("revocação", "Revocacao", True), ("taxa de falsos positivos", "TFP", True),
                               ("medida F1", "F1", True), ("alarmes falsos", "FP", False)]:
        if pct:
            print("     %-26s %7.2f%%  ->  %7.2f%%" % (rotulo, fixo[chave] * 100, cal[chave] * 100))
        else:
            print("     %-26s %8d  ->  %8d" % (rotulo, fixo[chave], cal[chave]))
    print()

print("Minutos em alarme por indicador, isoladamente (série completa de %s)\n" % inst)
decomposicao = []
for chave, col in COLS.items():
    cfg = LIMITES[chave]
    x = a["grade"][col]
    c_fixo = cfg["aviso"]
    c_cal = a["calibrados"][chave]["aviso"] if a["calibrados"][chave] else None
    disp_fixo = (x < c_fixo) if cfg["direcao"] == "abaixo" else (x > c_fixo)
    if c_cal is None:
        disp_cal = pd.Series(False, index=x.index)
    else:
        disp_cal = (x < c_cal) if cfg["direcao"] == "abaixo" else (x > c_cal)
    decomposicao.append(dict(
        Indicador=ROTULO_IND[chave],
        CorteFixo=c_fixo, CorteCalibrado=None if c_cal is None else round(c_cal, 2),
        MinutosAlarmeFixo=int(disp_fixo.fillna(False).sum()),
        MinutosAlarmeCalibrado=int(disp_cal.fillna(False).sum())))
pd.DataFrame(decomposicao)

Ganho da calibração em SQLDEV

  nível aviso
     revocação                    43.31%  ->    46.22%
     taxa de falsos positivos     33.79%  ->    13.18%
     medida F1                     3.11%  ->     7.86%
     alarmes falsos                 9080  ->      3543

  nível critico
     revocação                    19.48%  ->    29.65%
     taxa de falsos positivos     12.22%  ->     2.92%
     medida F1                     3.63%  ->    16.59%
     alarmes falsos                 3284  ->       784

Minutos em alarme por indicador, isoladamente (série completa de SQLDEV)



,Indicador,CorteFixo,CorteCalibrado,MinutosAlarmeFixo,MinutosAlarmeCalibrado
0,Expectativa de vida de página (s),1000,81.00,9135,1258
1,Taxa de acerto do buffer cache (%),97,99.76,165,1635
2,CPU do SQL Server (%),70,16.00,0,1247
3,Sessões bloqueadas,1,0.00,80,156


> **Leitura.** A decomposição mostra os dois erros independentes que o artigo descreve. A expectativa de vida de página disparava muito além do necessário e teve seu corte puxado para baixo, de 1.000 para 81 segundos; o processador quase nunca alcançava 70% e teve seu corte trazido para 16%, passando a capturar as saturações reais daquela instância. Um corte desceu, o outro desceu também, mas por motivos opostos — e por isso as duas medidas melhoraram juntas.

## 12. Terceira conclusão — as duas condições para calibrar

**Onde aparece no artigo:** terceiro parágrafo da seção 4.4 e terceiro parágrafo das considerações finais. É o achado que o artigo chama de "mais instrutivo".

Em LAB-SQL a calibração não apenas deixou de ajudar: piorou tudo, alarmando em 57,16% dos minutos. O artigo extrai daí duas condições de aplicabilidade, e esta célula mede cada uma.

**Dispersão.** Derivar um corte do percentil 95 pressupõe que exista um percentil 95 distinto do resto da distribuição. Numa instância ociosa, o processador fica em zero quase o tempo todo: o percentil 95 é zero, e a regra "alarme quando o processador passar de zero" dispara a qualquer sinal de vida. O desvio padrão da linha de base mede isso diretamente.

**Estacionariedade.** A expectativa de vida de página de uma instância sem pressão de memória não é uma variável que oscila em torno de um valor típico: é um contador que cresce desde a inicialização do serviço. A distribuição dela não descreve comportamento, descreve o tempo decorrido. A correlação de postos entre o valor e o instante da medição detecta isso: numa série que só cresce, ela vale exatamente 1. Percentis de uma rampa não são valores de corte, são marcas do relógio.

Repare na consequência prática: o percentil 5 dessa rampa, no laboratório, dá mais de um milhão de segundos — um valor que a série ultrapassa para baixo em quase todo o período seguinte, simplesmente porque o serviço foi reiniciado e o contador voltou a subir do começo.

In [19]:
print("Diagnóstico da linha de base: a calibração é aplicável?\n")
diagnostico = []
for inst in EXPERIMENTAIS:
    base = avaliacao[inst]["base"]
    for chave, col in COLS.items():
        x = base[col].dropna()
        instante = pd.Series(np.arange(len(x)), index=x.index)
        # correlação de postos entre valor e tempo (Spearman, por Pearson sobre os postos)
        rho = float(np.corrcoef(x.rank(), instante.rank())[0, 1]) if x.nunique() > 1 else np.nan
        cal = avaliacao[inst]["calibrados"][chave]
        diagnostico.append(dict(
            Instancia=inst, Indicador=ROTULO_IND[chave],
            ValoresDistintos=int(x.nunique()),
            DesvioPadrao=round(float(x.std()), 2),
            RhoComOTempo=round(rho, 3) if rho == rho else None,
            LimiarDerivado=None if cal is None else round(cal["aviso"], 2),
            Veredito=("sem dispersão — descartado" if cal is None
                      else "não estacionário" if rho is not None and rho > 0.95
                      else "aplicável")))

pd.DataFrame(diagnostico)

Diagnóstico da linha de base: a calibração é aplicável?



,Instancia,Indicador,ValoresDistintos,DesvioPadrao,RhoComOTempo,LimiarDerivado,Veredito
0,LAB-SQL,Expectativa de vida de página (s),10183,176376.03,1.000,1023315.00,não estacionário
1,LAB-SQL,Taxa de acerto do buffer cache (%),4,1.54,0.003,100.00,aplicável
2,LAB-SQL,CPU do SQL Server (%),10,0.40,0.047,0.00,aplicável
3,LAB-SQL,Sessões bloqueadas,2,0.01,0.017,0.00,aplicável
4,SQLDEV,Expectativa de vida de página (s),3019,3948.96,0.139,81.00,aplicável
5,SQLDEV,Taxa de acerto do buffer cache (%),258,1.62,0.008,99.76,aplicável
6,SQLDEV,CPU do SQL Server (%),45,4.50,-0.042,16.00,aplicável
7,SQLDEV,Sessões bloqueadas,4,0.07,0.001,0.00,aplicável


In [20]:
# Quantos minutos da série posterior ficam abaixo do percentil 5 derivado da rampa
a = avaliacao["LAB-SQL"]
corte = a["calibrados"]["ple_segundos"]["aviso"]
serie = a["grade"]["PageLifeExpectancy"]
posterior = serie[serie.index > a["base"].index.max()]
print("LAB-SQL — expectativa de vida de página")
print("   percentil 5 da linha de base (limiar derivado): %.0f s" % corte)
print("   minutos posteriores à base                    : %d" % len(posterior))
print("   destes, abaixo do limiar derivado             : %d (%.1f%%)"
      % ((posterior < corte).sum(), (posterior < corte).mean() * 100))
print()
print("   É esta a origem dos %d alarmes falsos do arranjo calibrado."
      % medidas(a["verdade"], a["arranjos"]["Limiar calibrado (aviso)"])["FP"])

LAB-SQL — expectativa de vida de página
   percentil 5 da linha de base (limiar derivado): 1023315 s
   minutos posteriores à base                    : 17046
   destes, abaixo do limiar derivado             : 15153 (88.9%)

   É esta a origem dos 15343 alarmes falsos do arranjo calibrado.


> **Leitura.** As duas condições falham no laboratório por motivos diferentes. Processador e sessões bloqueadas falham por **ausência de dispersão**: a linha de base tem tão poucos valores distintos que o percentil 95 coincide com o mínimo. A expectativa de vida de página falha por **não estacionariedade**: a correlação de postos com o tempo é 1,000, isto é, a série é estritamente crescente, e seu percentil 5 marca apenas onde o contador estava no início da janela. Em SQLDEV nenhuma das duas falhas ocorre, e é por isso que ali a mesma técnica funciona. A recomendação prática do artigo — exigir dispersão mínima e descartar indicadores não estacionários antes de calibrar — é exatamente o que esta tabela executaria como filtro automático.

---
## 13. Conferência final — valor publicado contra valor recalculado

Esta célula é a verificação do caderno inteiro. Cada linha traz um número tal como está no artigo e o mesmo número recalculado agora, com o veredito da comparação. É o que permite reabrir este arquivo daqui a seis meses e responder, sem depender de memória, se o texto ainda corresponde aos dados.

In [21]:
def pct(x): return "%.2f%%" % (x * 100)

conf = []
def confere(onde, descricao, publicado, recalculado):
    conf.append(dict(Local=onde, Valor=descricao, Publicado=str(publicado),
                     Recalculado=str(recalculado),
                     Veredito="confere" if str(publicado) == str(recalculado) else "DIVERGE"))

lab, dev = avaliacao["LAB-SQL"], avaliacao["SQLDEV"]
mf_lab = medidas(lab["verdade"], lab["arranjos"]["Limiar fixo (aviso)"])
mf_dev = medidas(dev["verdade"], dev["arranjos"]["Limiar fixo (aviso)"])
mfc_dev = medidas(dev["verdade"], dev["arranjos"]["Limiar fixo (crítico)"])
mc_dev = medidas(dev["verdade"], dev["arranjos"]["Limiar calibrado (aviso)"])
mcc_dev = medidas(dev["verdade"], dev["arranjos"]["Limiar calibrado (crítico)"])
mc_lab = medidas(lab["verdade"], lab["arranjos"]["Limiar calibrado (aviso)"])

confere("3.2", "minutos de série em LAB-SQL", "27.229", "{:,}".format(len(lab["grade"])).replace(",", "."))
confere("3.2", "minutos de série em SQLDEV", "27.218", "{:,}".format(len(dev["grade"])).replace(",", "."))
confere("3.2", "linha de base em LAB-SQL", "10.184",
        "{:,}".format(int((lab["grade"].index <= lab["inj"]["DataInicio"].min().floor("min")).sum())).replace(",", "."))
confere("3.2", "linha de base em SQLDEV", "10.903",
        "{:,}".format(int((dev["grade"].index <= dev["inj"]["DataInicio"].min().floor("min")).sum())).replace(",", "."))
confere("3.2", "minutos observacionais em produção", "26.516",
        "{:,}".format(len(grade("Prod"))).replace(",", "."))
confere("Resumo", "injeções concluídas em LAB-SQL", 98, len(lab["inj"]))
confere("Resumo", "injeções concluídas em SQLDEV", 88, len(dev["inj"]))
confere("4.3", "minutos com problema em LAB-SQL", 389, int(lab["verdade"].sum()))
confere("4.3", "minutos com problema em SQLDEV", 344, int(dev["verdade"].sum()))
confere("4.3", "prevalência em LAB-SQL", "1,43%", pct(lab["verdade"].sum() / len(lab["grade"])).replace(".", ","))
confere("4.3", "prevalência em SQLDEV", "1,26%", pct(dev["verdade"].sum() / len(dev["grade"])).replace(".", ","))
confere("Tab.1", "LAB-SQL fixo aviso — revocação", "13,37%", pct(mf_lab["Revocacao"]).replace(".", ","))
confere("Tab.1", "LAB-SQL fixo aviso — TFP", "0,25%", pct(mf_lab["TFP"]).replace(".", ","))
confere("Tab.1", "LAB-SQL fixo aviso — precisão", "43,70%", pct(mf_lab["Precisao"]).replace(".", ","))
confere("Tab.1", "LAB-SQL fixo aviso — alarmes falsos", 67, mf_lab["FP"])
confere("Tab.1", "LAB-SQL calibrado aviso — revocação", "97,43%", pct(mc_lab["Revocacao"]).replace(".", ","))
confere("Tab.1", "LAB-SQL calibrado aviso — TFP", "57,16%", pct(mc_lab["TFP"]).replace(".", ","))
confere("Tab.1", "LAB-SQL calibrado aviso — alarmes falsos", "15.343",
        "{:,}".format(mc_lab["FP"]).replace(",", "."))
confere("Tab.1", "SQLDEV fixo aviso — revocação", "43,31%", pct(mf_dev["Revocacao"]).replace(".", ","))
confere("Tab.1", "SQLDEV fixo aviso — TFP", "33,79%", pct(mf_dev["TFP"]).replace(".", ","))
confere("Tab.1", "SQLDEV fixo aviso — alarmes falsos", "9.080", "{:,}".format(mf_dev["FP"]).replace(",", "."))
confere("Tab.1", "SQLDEV fixo crítico — alarmes falsos", "3.284", "{:,}".format(mfc_dev["FP"]).replace(",", "."))
confere("Tab.1", "SQLDEV fixo crítico — F1", "3,63%", pct(mfc_dev["F1"]).replace(".", ","))
confere("Tab.1", "SQLDEV calibrado aviso — revocação", "46,22%", pct(mc_dev["Revocacao"]).replace(".", ","))
confere("Tab.1", "SQLDEV calibrado aviso — TFP", "13,18%", pct(mc_dev["TFP"]).replace(".", ","))
confere("Tab.1", "SQLDEV calibrado crítico — F1", "16,59%", pct(mcc_dev["F1"]).replace(".", ","))
confere("Tab.1", "SQLDEV calibrado crítico — alarmes falsos", 784, mcc_dev["FP"])
confere("Tab.1", "SQLDEV calibrado crítico — revocação", "29,65%", pct(mcc_dev["Revocacao"]).replace(".", ","))

for inst, md_, pub_fixo, pub_cal in [("LAB-SQL", "BLOQUEIO", "39,39%", "96,97%"),
                                     ("LAB-SQL", "CPU", "9,18%", "95,92%"),
                                     ("SQLDEV", "BLOQUEIO", "53,42%", "67,12%"),
                                     ("SQLDEV", "CPU", "22,43%", "27,10%"),
                                     ("SQLDEV", "IO", "37,50%", "26,04%"),
                                     ("SQLDEV", "MEMORIA", "73,53%", "82,35%")]:
    a = avaliacao[inst]
    sel = a["verdade"] & (a["modo"] == md_)
    n = int(sel.sum())
    confere("Tab.2", "%s %s — fixo" % (inst, ROTULO_MODO[md_]), pub_fixo,
            pct((sel & a["arranjos"]["Limiar fixo (aviso)"]).sum() / n).replace(".", ","))
    confere("Tab.2", "%s %s — calibrado" % (inst, ROTULO_MODO[md_]), pub_cal,
            pct((sel & a["arranjos"]["Limiar calibrado (aviso)"]).sum() / n).replace(".", ","))

confere("Tab.3", "LAB-SQL — PLE derivado (aviso)", "1.023.315",
        "{:,}".format(int(lab["calibrados"]["ple_segundos"]["aviso"])).replace(",", "."))
confere("Tab.3", "SQLDEV — PLE derivado (aviso)", "81", "%d" % dev["calibrados"]["ple_segundos"]["aviso"])
confere("Tab.3", "SQLDEV — buffer cache derivado (aviso)", "99,76",
        ("%.2f" % dev["calibrados"]["buffer_cache_hit_pct"]["aviso"]).replace(".", ","))
confere("Tab.3", "SQLDEV — CPU derivada (aviso)", "16", "%d" % dev["calibrados"]["cpu_sql_pct"]["aviso"])

q4 = pd.DataFrame(quadro4)
confere("Q.4", "LAB-SQL — PLE mediana", "1.298.246",
        "{:,}".format(int(q4.loc["Expectativa de vida de página (s)", "LAB-SQL"])).replace(",", "."))
confere("Q.4", "Prod — PLE mediana", "864", "%d" % q4.loc["Expectativa de vida de página (s)", "Prod"])
confere("Q.4", "Prod — conexões ativas medianas", "918", "%d" % q4.loc["Conexões ativas", "Prod"])
confere("Q.4", "Prod — pico de conexões", "2.204",
        "{:,}".format(int(q4.loc["Pico de conexões", "Prod"])).replace(",", "."))
confere("Q.4", "tipos de espera (LAB / DEV / Prod)", "65 / 127 / 133",
        " / ".join("%d" % q4.loc["Tipos de espera distintos", i] for i in INSTANCIAS))
confere("Q.1", "bancos e arquivos (LAB / DEV / Prod)", "12/32, 59/134, 69/154",
        ", ".join("%d/%d" % (r.Bancos, r.Arquivos) for r in portes.itertuples()))
confere("3.4", "registros com buffer cache acima de 100%", "4 de 22.670",
        "%d de %s" % (acima, "{:,}".format(registros).replace(",", ".")))

conferencia = pd.DataFrame(conf)
print("Conferidos: %d | conferem: %d | divergem: %d\n"
      % (len(conferencia), (conferencia["Veredito"] == "confere").sum(),
         (conferencia["Veredito"] == "DIVERGE").sum()))
conferencia

Conferidos: 51 | conferem: 49 | divergem: 2



,Local,Valor,Publicado,Recalculado,Veredito
0,3.2,minutos de série em LAB-SQL,27.229,27.229,confere
1,3.2,minutos de série em SQLDEV,27.218,27.218,confere
2,3.2,linha de base em LAB-SQL,10.184,10.184,confere
3,3.2,linha de base em SQLDEV,10.903,10.903,confere
4,3.2,minutos observacionais em produção,26.516,26.516,confere
5,Resumo,injeções concluídas em LAB-SQL,98,98,confere
6,Resumo,injeções concluídas em SQLDEV,88,88,confere
7,4.3,minutos com problema em LAB-SQL,389,389,confere
8,4.3,minutos com problema em SQLDEV,344,344,confere
9,4.3,prevalência em LAB-SQL,"1,43%","1,43%",confere


### O que fazer com as divergências

Se a coluna de veredito apontar divergência em alguma linha, há três causas possíveis, nesta ordem de probabilidade:

1. **A janela mudou.** A série continua crescendo. Se `FIM_DO_ARTIGO` for removido ou alargado, praticamente todos os números de contagem mudam. Este é o caso mais comum e não indica erro no artigo.
2. **O ambiente mudou.** Bancos criados ou removidos alteram o Quadro 1; uma reinicialização do serviço reinicia a expectativa de vida de página.
3. **O texto está desatualizado.** É o caso das duas divergências conhecidas neste caderno: a contagem de bancos no Quadro 1 e a contagem de registros com taxa de acerto acima de 100% na seção 3.4, ambas herdadas de uma extração anterior à última.